In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
episodes_with_id = pd.read_csv("../data/episodes_with_id.csv")
episodes_with_id = episodes_with_id[~episodes_with_id['category1'].isin(['religion', 'christianity'])]

In [3]:
episodes = []
num_dfs_appended = 0

for i in range(1, 11):
    for episode_id in episodes_with_id['episode_id']:
        file_path = f"../data/episodes/part_{i}/episode_{episode_id}.csv"
        try:
            df = pd.read_csv(file_path, usecols=lambda col: col in ['sentence', 'racialJustice', 'collectiveAction', 'collectiveActionMulti'])
            df['episode_id'] = episode_id  
            base_time = pd.to_datetime(episodes_with_id.loc[episodes_with_id['episode_id'] == episode_id, 'episodeDateLocalized'].values[0])
            df['time'] = base_time + pd.to_timedelta(df.index * 10, unit='s')
            if ((df['racialJustice'] == 1) & (df['collectiveAction'] == 0)).any():
                episodes.append(df) 
                num_dfs_appended += 1  
        except FileNotFoundError:
            continue

episodes = pd.concat(episodes, ignore_index=True)
episodes = episodes[episodes['episode_id'] != 1771]
num_dfs_appended = num_dfs_appended - 1
print(f"Number of episodes with at least one RJ+CA sentence: {num_dfs_appended}")

Number of episodes with at least one RJ+CA sentence: 6365


In [4]:
turns = []
num_turns_appended = 0

for episode_id in episodes['episode_id'].unique():
    file_path = f"../data/episodes/speaker_turns/episode_{episode_id}.csv"
    try:
        df_turn = pd.read_csv(file_path)
        df_turn['episode_id'] = episode_id
        turns.append(df_turn)
        num_turns_appended += 1
    except FileNotFoundError:
        continue

turns = pd.concat(turns, ignore_index=True)
print(f"Number of CSV files added to turns: {num_turns_appended}")

Number of CSV files added to turns: 6365


### Add metadata from speaker turns to sentences

In [ ]:
# import pandas as pd
# import numpy as np

# for n in episodes['episode_id'].unique():
#     # Filter rows for episode_id 
#     turns_episode_n = turns[turns['episode_id'] == n].copy()
#     episodes_episode_n = episodes[episodes['episode_id'] == n].copy()

#     # Perform a vectorized search: find matching turns
#     def find_speaker_and_features(sentence):
#         match = turns_episode_n.loc[turns_episode_n['turnText'].str.contains(sentence, case=False, na=False, regex=False), 
#                                     ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean', 
#                                      'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean', 
#                                      'startTime', 'endTime', 'duration', 'turnCount']]
#         if not match.empty:
#             return match.iloc[0]
#         else:
#             # Return NaNs for the additional columns if no match is found
#             return pd.Series([np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan], 
#                              index=['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean', 
#                                     'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean', 
#                                     'startTime', 'endTime', 'duration', 'turnCount'])

#     # Apply vectorized function
#     episodes_episode_n[['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean', 
#                         'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean', 
#                         'startTime', 'endTime', 'duration', 'turnCount']] = episodes_episode_n['sentence'].apply(find_speaker_and_features)

#     # Update original DataFrame
#     episodes.loc[episodes['episode_id'] == n, ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 
#                                                 'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 
#                                                 'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount']] = \
#         episodes_episode_n[['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean', 
#                             'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean', 
#                             'startTime', 'endTime', 'duration', 'turnCount']]

# # Check the updated episodes DataFrame
# episodes[['sentence', 'speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean', 
#           'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean', 'startTime', 
#           'endTime', 'duration', 'turnCount', 'episode_id']]

### improved version of searching for metadata that only considers future speaker turns

In [ ]:
# for n in episodes['episode_id'].unique():
#     # Filter rows for episode_id
#     turns_episode_n = turns[turns['episode_id'] == n].copy()
#     episodes_episode_n = episodes[episodes['episode_id'] == n].copy()

#     # Track the highest turnCount assigned for this episode_id
#     max_turn_count_assigned = -1

#     # Perform a vectorized search: find matching turns
#     def find_speaker_and_features(sentence, max_turn_count_assigned):
#         match = turns_episode_n.loc[turns_episode_n['turnText'].str.contains(sentence, case=False, na=False, regex=False),
#                                     ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean',
#                                      'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean',
#                                      'startTime', 'endTime', 'duration', 'turnCount']]

#         if not match.empty:
#             # Loop through matches and pick the first available turnCount that hasn't been assigned
#             for idx, row in match.iterrows():
#                 # Only assign turnCount if it is greater than or equal to max_turn_count_assigned
#                 if row['turnCount'] >= max_turn_count_assigned:
#                     max_turn_count_assigned = row['turnCount']  # Update the highest turnCount assigned
#                     return row, max_turn_count_assigned  # Return the first valid match and the updated max_turn_count_assigned
#             # If no valid match is found, return NaNs
#             return pd.Series([np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
#                              index=['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean',
#                                     'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean',
#                                     'startTime', 'endTime', 'duration', 'turnCount']), max_turn_count_assigned
#         else:
#             # If no match is found, return NaNs
#             return pd.Series([np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
#                              index=['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean',
#                                     'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean',
#                                     'startTime', 'endTime', 'duration', 'turnCount']), max_turn_count_assigned

#     # Apply vectorized function to assign the corresponding features
#     result = episodes_episode_n['sentence'].apply(find_speaker_and_features, args=(max_turn_count_assigned,))

#     # Unpack the results into two columns
#     episodes_episode_n[['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean',
#                         'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean',
#                         'startTime', 'endTime', 'duration', 'turnCount']], max_turn_count_assigned = zip(*result)

#     # Update the original DataFrame
#     episodes.loc[episodes['episode_id'] == n, ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean',
#                                                 'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean',
#                                                 'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount']] = \
#         episodes_episode_n[['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean',
#                             'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean',
#                             'startTime', 'endTime', 'duration', 'turnCount']]

In [57]:
# Iterate over each unique episode_id
for n in episodes['episode_id'].unique():
    # Filter data for the current episode
    turns_episode_n = turns[turns['episode_id'] == n].copy()
    episodes_episode_n = episodes[episodes['episode_id'] == n].copy()

    # Reset max_turn_count_assigned for each episode
    global max_turn_count_assigned  
    max_turn_count_assigned = -1  # Ensuring it starts fresh per episode_id

    def find_speaker_and_features(sentence):
        global max_turn_count_assigned  # Ensure updates persist across function calls

        match = turns_episode_n.loc[
            turns_episode_n['turnText'].str.contains(sentence, case=False, na=False, regex=False),
            ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean',
             'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean',
             'startTime', 'endTime', 'duration', 'turnCount']
        ]

        if not match.empty:
            for idx, row in match.iterrows():
                if row['turnCount'] >= max_turn_count_assigned:
                    max_turn_count_assigned = row['turnCount']  # Update max_turn_count_assigned
                    return row  # Return the first valid match

        # If no valid match is found, return NaNs
        return pd.Series([np.nan] * 12, 
                         index=['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 
                                'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 
                                'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount'])

    # Apply the function and ensure turnCount updates
    episodes_episode_n[['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean',
                        'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean',
                        'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount']] = \
        episodes_episode_n['sentence'].apply(find_speaker_and_features)

    # Update the main DataFrame
    episodes.loc[episodes['episode_id'] == n, ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean',
                                                'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean',
                                                'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount']] = \
        episodes_episode_n[['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean',
                            'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean',
                            'startTime', 'endTime', 'duration', 'turnCount']]

Version 2 only attempts to match sentences to speaker turns if the sentence is longer than 2 words

In [5]:
# Iterate over each unique episode_id
for n in episodes['episode_id'].unique():
    # Filter data for the current episode
    turns_episode_n = turns[turns['episode_id'] == n].copy()
    episodes_episode_n = episodes[episodes['episode_id'] == n].copy()

    # Reset max_turn_count_assigned for each episode
    global max_turn_count_assigned  
    max_turn_count_assigned = -1  # Ensuring it starts fresh per episode_id

    def find_speaker_and_features(sentence):
        global max_turn_count_assigned  # Ensure updates persist across function calls
        
        # Check if the sentence has at least 3 words
        if len(str(sentence).split()) < 3:
            return pd.Series([np.nan] * 12, 
                             index=['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 
                                    'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 
                                    'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount'])

        match = turns_episode_n.loc[
            turns_episode_n['turnText'].str.contains(sentence, case=False, na=False, regex=False),
            ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean',
             'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean',
             'startTime', 'endTime', 'duration', 'turnCount']
        ]

        if not match.empty:
            for idx, row in match.iterrows():
                if row['turnCount'] >= max_turn_count_assigned:
                    max_turn_count_assigned = row['turnCount']  # Update max_turn_count_assigned
                    return row  # Return the first valid match

        # If no valid match is found, return NaNs
        return pd.Series([np.nan] * 12, 
                         index=['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 
                                'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 
                                'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount'])

    # Apply the function and ensure turnCount updates
    episodes_episode_n[['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean',
                        'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean',
                        'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount']] = \
        episodes_episode_n['sentence'].apply(find_speaker_and_features)

    # Update the main DataFrame
    episodes.loc[episodes['episode_id'] == n, ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean',
                                                'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean',
                                                'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount']] = \
        episodes_episode_n[['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean',
                            'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean',
                            'startTime', 'endTime', 'duration', 'turnCount']]

This works but it takes 50 min.

In [ ]:
# import numpy as np
# import pandas as pd

# for n in episodes['episode_id'].unique():
#     # Filter rows for episode_id
#     turns_episode_n = turns[turns['episode_id'] == n].copy()
#     episodes_episode_n = episodes[episodes['episode_id'] == n].copy()

#     # Track the highest turnCount assigned for this episode_id
#     max_turn_count_assigned = -1

#     # Iterate over each row to ensure proper tracking
#     for idx, row in episodes_episode_n.iterrows():
#         sentence = row['sentence']
        
#         # Find matching turn
#         match = turns_episode_n.loc[
#             turns_episode_n['turnText'].str.contains(sentence, case=False, na=False, regex=False),
#             ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean',
#              'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean',
#              'startTime', 'endTime', 'duration', 'turnCount']
#         ]

#         if not match.empty:
#             for match_idx, match_row in match.iterrows():
#                 if match_row['turnCount'] >= max_turn_count_assigned:
#                     max_turn_count_assigned = match_row['turnCount']  # Update turn count
                    
#                     # Assign all the matching metadata using .loc
#                     episodes_episode_n.loc[idx, 
#                         ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 
#                          'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 
#                          'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount']
#                     ] = match_row.values
#                     break  # Assign only the first valid match

#         else:
#             # If no match, assign NaN
#             episodes_episode_n.loc[idx, 
#                 ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 
#                  'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 
#                  'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount']
#             ] = np.nan

#     # Update the original DataFrame with the modified values
#     episodes.loc[episodes['episode_id'] == n, 
#         ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 
#          'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 
#          'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount']
#     ] = episodes_episode_n[
#         ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 
#          'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 
#          'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount']
#     ]

In [8]:
# Calculate the number of rows where speaker is None
num_none_speakers = episodes['speaker'].isna().sum()

# Calculate the percentage of rows where speaker is None
percentage_none_speakers = (num_none_speakers / len(episodes)) * 100

print(f"Number of rows with speaker = None: {num_none_speakers}")
print(f"Percentage of rows with speaker = None: {percentage_none_speakers:.2f}%")

Number of rows with speaker = None: 302732
Percentage of rows with speaker = None: 19.69%


In [ ]:
# episodes.loc[~((episodes['collectiveAction'] == 0) & (episodes['racialJustice'] == 1)), 'collectiveActionMulti'] = np.nan

In [ ]:
# episodes.to_csv('../data/sentences_with_metadata.csv', index=False)

In [10]:
episodes.to_csv('../data/sentences_with_metadata_order_3ormore.csv', index=False)

# Making new speaker turns 

In [5]:
episodes = pd.read_csv('../data/sentences_with_metadata_order.csv')

In [11]:
episodes = episodes.drop(columns=['time'])

In [12]:
episodes.columns

Index(['sentence', 'collectiveAction', 'racialJustice',
       'collectiveActionMulti', 'episode_id', 'speaker', 'newSpeaker',
       'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean', 'mfcc4_sma3Mean',
       'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean',
       'startTime', 'endTime', 'duration', 'turnCount'],
      dtype='object')

In [13]:
max_rows = episodes.groupby(['episode_id', 'speaker', 'turnCount']).size().max()
print(f"Maximum number of rows with the same episode_id, speaker, and turnCount: {max_rows}")

Maximum number of rows with the same episode_id, speaker, and turnCount: 1825


In [14]:
episodes_with_id = pd.read_csv('../data/episodes_with_id.csv')
# episodes_with_id = episodes_with_id[~episodes_with_id['category1'].isin(['religion', 'christianity'])]
# episodes = episodes[episodes['episode_id'] != 1771]
len(episodes_with_id)

10496

In [15]:
with open('../data/good_episode_ids.txt', 'r') as file:
    good_episode_ids = file.read().splitlines()
episodes_with_id['episode_id'] = episodes_with_id['episode_id'].astype(str)
good_episode_ids = [str(eid) for eid in good_episode_ids]
episodes_with_id = episodes_with_id[episodes_with_id['episode_id'].isin(good_episode_ids)]
len(episodes_with_id)

6365

### Fixing metadata for single-speaker episodes

In [16]:
count_totalSpLabels_1 = episodes_with_id[episodes_with_id['totalSpLabels'] == 1].shape[0]
print(f"Number of rows with totalSpLabels = 1: {count_totalSpLabels_1}")

Number of rows with totalSpLabels = 1: 2772


In [17]:
count_numMainSpeakers_1 = episodes_with_id[episodes_with_id['numMainSpeakers'] == 1].shape[0]
print(f"Number of rows with numMainSpeakers = 1: {count_numMainSpeakers_1}")

Number of rows with numMainSpeakers = 1: 2772


In [18]:
episodes['episode_id'] = episodes['episode_id'].astype(str)

In [19]:
unique_episode_ids = episodes_with_id.loc[episodes_with_id['numMainSpeakers'] == 1, 'episode_id'].astype(str).unique()
filtered_episodes = episodes[episodes['episode_id'].isin(unique_episode_ids)]
num_nan_speakers = filtered_episodes['speaker'].isna().sum()
print(num_nan_speakers)

18690


For the 584 rows without metadata in the 2772 single-speaker episode, inherit metadata from another row with the same episode_id.

In [20]:
filtered_episodes[['speaker', 'newSpeaker',
       'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean', 'mfcc4_sma3Mean',
       'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean',
       'startTime', 'endTime', 'duration', 'turnCount']] = (
    filtered_episodes.groupby('episode_id')[['speaker', 'newSpeaker',
       'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean', 'mfcc4_sma3Mean',
       'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean',
       'startTime', 'endTime', 'duration', 'turnCount']]
    .transform(lambda x: x.ffill().bfill())
)

/var/folders/z_/3vpzgjpj4dv90c8dxq0x3ww80000gn/T/ipykernel_93931/870383090.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_episodes[['speaker', 'newSpeaker',


In [21]:
episodes.update(filtered_episodes)

In [77]:
# episodes.to_csv('../data/sentences_with_metadata_order_singles_fixed.csv', index=False)

In [22]:
# Calculate the number of rows where speaker is None
num_none_speakers = episodes['speaker'].isna().sum()

# Calculate the percentage of rows where speaker is None
percentage_none_speakers = (num_none_speakers / len(episodes)) * 100

print(f"Number of rows with speaker = None: {num_none_speakers}")
print(f"Percentage of rows with speaker = None: {percentage_none_speakers:.2f}%")

Number of rows with speaker = None: 284042
Percentage of rows with speaker = None: 18.48%


### Checking for episodes with incorrect turn sequences

Here 1, 1, 2, 3 is fine but 1, 1, 3 is not (we ignore NaN)

In [23]:
def is_valid_sequence(turn_counts):
    # Remove NaN values for checking the sequence
    valid_turn_counts = turn_counts.dropna()
    
    # Check if the sequence has any unexpected jumps
    if len(valid_turn_counts) > 1:
        diff = valid_turn_counts.diff().dropna()
        # If any difference is not 0 or 1, the sequence is invalid
        if any((diff != 1) & (diff != 0)):
            return False
    return True

# Group by episode_id and apply the validation
invalid_episode_count = 0

for episode_id, group in episodes.groupby('episode_id'):
    if not is_valid_sequence(group['turnCount']):
        invalid_episode_count += 1

print(f"Number of episodes with an incorrect sequence: {invalid_episode_count}")

Number of episodes with an incorrect sequence: 3384


Here 1, 1, 3 is fine (we ignore NaN)

In [24]:
def is_non_decreasing_sequence(turn_counts):
    # Remove NaN values for checking the sequence
    valid_turn_counts = turn_counts.dropna()
    
    # Check if the sequence is in non-decreasing order
    if len(valid_turn_counts) > 1:
        diff = valid_turn_counts.diff().dropna()
        # If any difference is negative, the sequence is invalid
        if any(diff < 0):
            return False
    return True

# Group by episode_id and apply the validation
invalid_episode_count = 0

for episode_id, group in episodes.groupby('episode_id'):
    if not is_non_decreasing_sequence(group['turnCount']):
        invalid_episode_count += 1

print(f"Number of episodes with a non-growing sequence: {invalid_episode_count}")

Number of episodes with a non-growing sequence: 0


### Matching NaN to turn before/after

In [ ]:
# episodes = pd.read_csv('../data/sentences_with_metadata_order_singles_fixed.csv')

In [25]:
nan_speaker_episodes = episodes[episodes['speaker'].isna()]
duplicate_sentences = nan_speaker_episodes[nan_speaker_episodes.duplicated(subset=['sentence'], keep=False)]

In [26]:
speaker_counts = episodes['speaker'].value_counts()
print(speaker_counts)

speaker
SPEAKER_00                                                    792999
SPEAKER_01                                                    340040
SPEAKER_02                                                     67092
SPEAKER_03                                                     18742
SPEAKER_00, SPEAKER_01                                         10427
SPEAKER_01, SPEAKER_00                                          7972
SPEAKER_04                                                      7273
SPEAKER_05                                                      2096
SPEAKER_02, SPEAKER_00, SPEAKER_01                              1629
SPEAKER_00, SPEAKER_02, SPEAKER_01                               620
SPEAKER_06                                                       547
SPEAKER_02, SPEAKER_01, SPEAKER_00                               485
SPEAKER_01, SPEAKER_02, SPEAKER_00                               479
SPEAKER_00, SPEAKER_01, SPEAKER_02                               430
SPEAKER_00, SPEAKER_01, SP

In [27]:
speaker_counts = episodes['newSpeaker'].value_counts()
print(speaker_counts)

newSpeaker
SPEAKER_00    804958
SPEAKER_01    349103
SPEAKER_02     69918
SPEAKER_03     19163
SPEAKER_04      7287
SPEAKER_05      2099
SPEAKER_06       551
SPEAKER_08        61
SPEAKER_07        51
SPEAKER_09        46
Name: count, dtype: int64


If a NaN is between two rows with the same episode_id and same turnCount, it inherits the metadata.

In [28]:
for i in range(1, len(episodes) - 1): 
    try:
        if pd.isna(episodes.loc[i, 'speaker']):  # Check if 'speaker' is NaN
            prev_row = episodes.loc[i - 1]  # Previous row
            next_row = episodes.loc[i + 1]  # Next row
            
            if (prev_row['episode_id'] == next_row['episode_id'] and
                prev_row['speaker'] == next_row['speaker'] and
                prev_row['newSpeaker'] == next_row['newSpeaker'] and
                prev_row['turnCount'] == next_row['turnCount']):
            
                # Assign the metadata from the previous row to the current row
                columns_to_copy = [
                    'speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 
                    'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 
                    'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount'
                ]
                
                for column in columns_to_copy:
                    episodes.loc[i, column] = prev_row[column]

    except KeyError:  # Handle case where the index does not exist
        print(f"Skipping index {i} due to missing value.")
        continue  # Continue to the next iteration

Skipping index 200676 due to missing value.
Skipping index 200677 due to missing value.
Skipping index 200678 due to missing value.
Skipping index 200679 due to missing value.
Skipping index 200680 due to missing value.
Skipping index 200681 due to missing value.
Skipping index 200682 due to missing value.
Skipping index 200683 due to missing value.
Skipping index 200684 due to missing value.
Skipping index 200685 due to missing value.
Skipping index 200686 due to missing value.
Skipping index 200687 due to missing value.
Skipping index 200688 due to missing value.
Skipping index 200689 due to missing value.
Skipping index 200690 due to missing value.
Skipping index 200691 due to missing value.
Skipping index 200692 due to missing value.
Skipping index 200693 due to missing value.
Skipping index 200694 due to missing value.
Skipping index 200695 due to missing value.
Skipping index 200696 due to missing value.
Skipping index 200697 due to missing value.
Skipping index 200698 due to mis

In [29]:
# Calculate the number of rows where speaker is None
num_none_speakers = episodes['speaker'].isna().sum()

# Calculate the percentage of rows where speaker is None
percentage_none_speakers = (num_none_speakers / len(episodes)) * 100

print(f"Number of rows with speaker = None: {num_none_speakers}")
print(f"Percentage of rows with speaker = None: {percentage_none_speakers:.2f}%")

Number of rows with speaker = None: 268959
Percentage of rows with speaker = None: 17.50%


If n consectuive NaN rows are between two rows with the same episode_id and same turnCount, they inherit the metadata.

This code takes care of n, nan (x times), n

In [32]:
for i in range(len(episodes) - 4):  # Check for sequences with 4 or more rows
    # Check for patterns with 3 to 5 NaN values
    for n in range(2, 10):  # Loop over the number of NaN values (9 to 20 NaNs)
        if (not pd.isna(episodes.loc[i, 'turnCount']) and  # The first value is not NaN
            all(pd.isna(episodes.loc[i + j, 'turnCount']) for j in range(1, n + 1)) and  # The next n values are NaN
            not pd.isna(episodes.loc[i + n + 1, 'turnCount']) and  # The last value after NaNs is not NaN
            episodes.loc[i, 'turnCount'] == episodes.loc[i + n + 1, 'turnCount']):  # Matching turnCount between the first and last
            print(f"Pattern found at index {i} with {n} NaNs:")
            print(episodes.loc[i:i + n + 1])  # Print the detected pattern

            # Copy the turnCount and metadata from the first or last valid row
            columns_to_copy = [
                'speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean',
                'mfcc3_sma3Mean', 'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean',
                'F1frequency_sma3nzMean', 'startTime', 'endTime', 'duration', 'turnCount'
            ]
            
            # Propagate the values from the valid row (either i or i + n + 1) to the NaN rows
            for j in range(i + 1, i + n + 1):  # Loop over the NaN rows
                for column in columns_to_copy:
                    episodes.loc[j, column] = episodes.loc[i, column]  # Copy metadata from the first valid row

Pattern found at index 177 with 2 NaNs:
                                                                                                                sentence  \
177  It's an opportunity to get to know your families and introduce them to each other so that you can truly build them.   
178                                                                                                        - Absolutely.   
179                                                                                                             - Right?   
180                                          And we did that for the first time with one of our schools a few years ago.   

     collectiveAction  racialJustice  collectiveActionMulti episode_id  \
177                 0              0                    3.0          1   
178                 1              0                    NaN          1   
179                 1              0                    NaN          1   
180                 1              0       

KeyError: 200676

In [ ]:
# Calculate the number of rows where speaker is None
num_none_speakers = episodes['speaker'].isna().sum()

# Calculate the percentage of rows where speaker is None
percentage_none_speakers = (num_none_speakers / len(episodes)) * 100

print(f"Number of rows with speaker = None: {num_none_speakers}")
print(f"Percentage of rows with speaker = None: {percentage_none_speakers:.2f}%")

Number of rows with speaker = None: 325259
Percentage of rows with speaker = None: 21.16%


If a NaN is between two rows with the same episode_id and turnCount-1 and turnCount+1, it inherits the metadata of the row from turns with that episode_id and turnCount

In [261]:
# Loop through each row in episodes where speaker is NaN
for index, row in episodes[episodes['speaker'].isna()].iterrows():
    episode_id = row['episode_id']
    turn_count = row['turnCount']
    
    # Check the previous row and next row for matching episode_id and consecutive turnCount
    if index > 0 and index < len(episodes) - 1:
        prev_row = episodes.iloc[index - 1]
        next_row = episodes.iloc[index + 1]

        #print(f"prev: {prev_row['turnCount']} -- current: {turn_count} -- next: {next_row['turnCount']}")
        
        #if (prev_row['episode_id'] == episode_id and prev_row['turnCount'] == turn_count - 1) and (next_row['episode_id'] == episode_id and next_row['turnCount'] == turn_count + 1):
        if (prev_row['episode_id'] == episode_id and next_row['episode_id'] == episode_id and prev_row['turnCount'] == next_row['turnCount']-2):   
            print(f"Episode ID: {episode_id}, Turn Count: {turn_count}")
            
            # Get the corresponding row in episodes_turns
            metadata_row = turns[(turns['episode_id'] == episode_id) & (turns['turnCount'] == prev_row['turnCount']+1)]
            
            if not metadata_row.empty:
                # Assign metadata from the episodes_turns row to the current row
                for col in ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean', 
                            'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean', 
                            'startTime', 'endTime', 'duration', 'turnCount']:
                    episodes.at[index, col] = metadata_row[col].values[0]

                print(f"Metadata assigned to row {index} :: turn={metadata_row['turnCount']}.")

In [262]:
# Calculate the number of rows where speaker is None
num_none_speakers = episodes['speaker'].isna().sum()

# Calculate the percentage of rows where speaker is None
percentage_none_speakers = (num_none_speakers / len(episodes)) * 100

print(f"Number of rows with speaker = None: {num_none_speakers}")
print(f"Percentage of rows with speaker = None: {percentage_none_speakers:.2f}%")

Number of rows with speaker = None: 325259
Percentage of rows with speaker = None: 21.16%


In [ ]:
#episodes.to_csv('../data/sentences_with_metadata_order_singles_fixed_and_middles.csv', index=False)

Deal with cases such as 8, NaN, NaN, NaN, 12

In [ ]:
# # Loop through each row in episodes where speaker is NaN
# for index, row in episodes[episodes['speaker'].isna()].iterrows():
#     episode_id = row['episode_id']
#     turn_count = row['turnCount']
    
#     # Check the previous row and next row for matching episode_id and consecutive turnCount
#     if index > 0 and index < len(episodes) - 1:
#         prev_row = episodes.iloc[index - 1]
#         next_row = episodes.iloc[index + 1]

#         # Ensure the episode_id matches and that the turnCount difference is greater than 1
#         if (prev_row['episode_id'] == episode_id and next_row['episode_id'] == episode_id and prev_row['turnCount'] == next_row['turnCount'] - 2):
#             print(f"Episode ID: {episode_id}, Turn Count: {turn_count}")
            
#             # Calculate the number of missing values (NaNs) based on the difference between turnCount
#             missing_turns = next_row['turnCount'] - prev_row['turnCount'] - 1
            
#             # Count how many consecutive NaNs exist
#             nan_count = episodes.loc[index + 1:index + missing_turns, 'speaker'].isna().sum()

#             # Check if the number of consecutive NaNs matches the expected difference
#             if nan_count == missing_turns:
#                 # Assign consecutive turn counts for NaN values between prev_row and next_row
#                 for i in range(1, missing_turns + 1):
#                     episodes.at[index + i, 'turnCount'] = prev_row['turnCount'] + i

#                 # Now fill the missing speaker and metadata info based on the turn counts
#                 for i in range(1, missing_turns + 1):
#                     # Get the corresponding row in episodes_turns for the current missing turn count
#                     metadata_row = turns[(turns['episode_id'] == episode_id) & (turns['turnCount'] == prev_row['turnCount'] + i)]
                    
#                     if not metadata_row.empty:
#                         # Assign metadata from the episodes_turns row to the current missing row
#                         for col in ['speaker', 'newSpeaker', 'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean', 
#                                     'mfcc4_sma3Mean', 'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean', 
#                                     'startTime', 'endTime', 'duration', 'turnCount']:
#                             episodes.at[index + i, col] = metadata_row[col].values[0]

#                         print(f"Metadata assigned to row {index + i} :: turn={metadata_row['turnCount']}.")
#             else:
#                 print(f"Warning: Number of consecutive NaNs ({nan_count}) does not match the expected difference ({missing_turns}).")

In [265]:
# Calculate the number of rows where speaker is None
num_none_speakers = episodes['speaker'].isna().sum()

# Calculate the percentage of rows where speaker is None
percentage_none_speakers = (num_none_speakers / len(episodes)) * 100

print(f"Number of rows with speaker = None: {num_none_speakers}")
print(f"Percentage of rows with speaker = None: {percentage_none_speakers:.2f}%")

Number of rows with speaker = None: 325259
Percentage of rows with speaker = None: 21.16%


### Concatenating consecutive sentences from the same speaker turn

In [270]:
pd.set_option('display.max_colwidth', None)

In [49]:
# # Group by 'episode_id' and 'group', ensuring only consecutive turns are grouped
# aggregated_df = episodes.groupby(['episode_id', 'turnCount'], as_index=False).agg(
#     # Concatenate the 'sentence' column for all rows in each group
#     sentence=('sentence', ' '.join),
    
#     # For 'racialJustice', if any row in the group is 1, the result should be 1
#     racialJustice=('racialJustice', 'max'),
    
#     # For 'collectiveAction', if any row in the group is 0, the result should be 0
#     collectiveAction=('collectiveAction', 'min'),
    
#     # For 'collectiveActionMulti', collect all non-null values in a list
#     collectiveActionMulti=('collectiveActionMulti', lambda x: x.dropna().tolist() if not x.dropna().empty else None)
# )

In [ ]:
# # Filter out rows where 'turnCount' is NaN before grouping
# filtered_df = episodes.dropna(subset=['turnCount'])

# # Group by 'episode_id' and 'turnCount'
# aggregated_df = filtered_df.groupby(['episode_id', 'turnCount'], as_index=False).agg(
#     sentence=('sentence', ' '.join),  # Concatenate sentences
#     racialJustice=('racialJustice', 'max'),  # Take the max (if any row is 1, result is 1)
#     collectiveAction=('collectiveAction', 'min'),  # Take the min (if any row is 0, result is 0)
#     collectiveActionMulti=('collectiveActionMulti', lambda x: x.dropna().tolist() if not x.dropna().empty else None)
# )

# # Append back rows where 'turnCount' was NaN (unmodified)
# aggregated_df = pd.concat([aggregated_df, episodes[episodes['turnCount'].isna()]], ignore_index=True)

In [ ]:
# # Create a copy of the original DataFrame to retain all columns
# modified_episodes = episodes.copy()

# # Identify columns that are not used for grouping
# non_grouped_cols = [col for col in episodes.columns if col not in ['episode_id', 'turnCount', 'sentence', 'racialJustice', 'collectiveAction', 'collectiveActionMulti']]

# # Filter out rows where 'turnCount' is NaN before grouping
# filtered_df = modified_episodes.dropna(subset=['turnCount'])

# # Group by 'episode_id' and 'turnCount', applying relevant aggregations
# aggregated_df = filtered_df.groupby(['episode_id', 'turnCount'], as_index=False).agg({
#     'sentence': ' '.join,  # Concatenate sentences
#     'racialJustice': 'max',  # Take the max (if any row is 1, result is 1)
#     'collectiveAction': 'min',  # Take the min (if any row is 0, result is 0)
#     'collectiveActionMulti': lambda x: x.dropna().tolist() if not x.dropna().empty else None
# })

# # Ensure the non-grouped columns are preserved
# for col in non_grouped_cols:
#     aggregated_df[col] = filtered_df.groupby(['episode_id', 'turnCount'])[col].first().reset_index(drop=True)

# # Append back rows where 'turnCount' was NaN (unmodified)
# final_df = pd.concat([aggregated_df, modified_episodes[modified_episodes['turnCount'].isna()]], ignore_index=True)

# # Ensure column order matches the original DataFrame
# final_df = final_df[modified_episodes.columns]

This is not inserting NaN backs in the right place.

In [36]:
episodes.columns

Index(['sentence', 'collectiveAction', 'racialJustice',
       'collectiveActionMulti', 'episode_id', 'speaker', 'newSpeaker',
       'mfcc1_sma3Mean', 'mfcc2_sma3Mean', 'mfcc3_sma3Mean', 'mfcc4_sma3Mean',
       'F0semitoneFrom27_5Hz_sma3nzMean', 'F1frequency_sma3nzMean',
       'startTime', 'endTime', 'duration', 'turnCount'],
      dtype='object')

In [279]:
# Filter out rows where 'turnCount' is NaN before grouping
filtered_df = episodes.dropna(subset=['turnCount'])

# Group by 'episode_id' and 'turnCount'
aggregated_df = filtered_df.groupby(['episode_id', 'turnCount'], as_index=False).agg(
    sentence=('sentence', ' '.join),  # Concatenate sentences
    racialJustice=('racialJustice', 'max'),  # Take the max (if any row is 1, result is 1)
    collectiveAction=('collectiveAction', 'min'),  # Take the min (if any row is 0, result is 0)
    collectiveActionMulti=('collectiveActionMulti', lambda x: x.dropna().tolist() if not x.dropna().empty else None),
    episode_id=('episode_id', 'first'),  # Keep the first value (assuming the same for all rows)
    speaker=('speaker', 'first'),  # Keep the first value
    newSpeaker=('newSpeaker', 'first'),  # Keep the first value
    mfcc1_sma3Mean=('mfcc1_sma3Mean', 'first'),  # Keep the first value
    mfcc2_sma3Mean=('mfcc2_sma3Mean', 'first'),  # Keep the first value
    mfcc3_sma3Mean=('mfcc3_sma3Mean', 'first'),  # Keep the first value
    mfcc4_sma3Mean=('mfcc4_sma3Mean', 'first'),  # Keep the first value
    F0semitoneFrom27_5Hz_sma3nzMean=('F0semitoneFrom27_5Hz_sma3nzMean', 'first'),  # Keep the first value
    F1frequency_sma3nzMean=('F1frequency_sma3nzMean', 'first'),  # Keep the first value
    startTime=('startTime', 'first'),  # Keep the first value
    endTime=('endTime', 'first'),  # Keep the first value
    duration=('duration', 'first'),  # Keep the first value
    turnCount=('turnCount', 'first')  # Keep the first value
)

# Select rows where 'turnCount' is NaN, keeping the original order
na_rows = episodes[episodes['turnCount'].isna()]

# Concatenate the rows with NaN 'turnCount' back into the aggregated DataFrame
# Ensure the rows with NaN 'turnCount' stay in their original position
aggregated_df = pd.concat([aggregated_df, na_rows]).sort_index()

# Reset index after concatenation
aggregated_df = aggregated_df.reset_index(drop=True)

In [280]:
len(aggregated_df) # 1,537,279 original vs 515,654 aggregated vs 392,492 speaker turns

442464

In [ ]:
#aggregated_df = pd.DataFrame()